# Minimum Variance Portfolio

**Objective key:** `min_variance` &nbsp;·&nbsp; **Family:** Classical &nbsp;·&nbsp; **Speed:** Fast

## Algorithm

Finds the portfolio that minimises total variance `wᵀΣw` subject to weights summing  
to 1 and weight bounds. Expected returns `mu` are **not** used in the objective —  
useful when return estimates are noisy or unreliable.

## Papers

- **Foundational:** Markowitz (1952) — *Portfolio Selection* — Journal of Finance  
  https://doi.org/10.1111/j.1540-6261.1952.tb01525.x
- **Modern (covariance estimation):** Ledoit & Wolf (2020) — *Analytical Nonlinear Shrinkage* — Annals of Statistics  
  https://arxiv.org/abs/1910.13597 (arXiv, open access)

## Assumptions

- `Sigma` is annualised; `mu` is passed but ignored in the objective.
- `weight_min=0.005`, `weight_max=0.20`, `seed=42`.
- Sample covariance is used here; production code may apply Ledoit–Wolf shrinkage.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 10
mu = rng.uniform(0.06, 0.22, n)  # not used in objective, but required by API
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print("Covariance diag (variances):", np.diag(Sigma).round(4))

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="min_variance",
    asset_names=asset_names,
    weight_min=0.005,
    weight_max=0.20,
    seed=42,
)

print(f"\nObjective:       {result.objective}")
print(f"Portfolio variance (ann.):  {result.volatility**2:.6f}")
print(f"Portfolio volatility (ann.): {result.volatility:.4f}")
print(f"Expected return:            {result.expected_return:.4f}")
print(f"Sharpe ratio:               {result.sharpe_ratio:.4f}")
print("\nWeights:")
for name, w in zip(asset_names, result.weights):
    print(f"  {name}: {w:.4f}")

In [ ]:
# Min-variance should have lower vol than equal-weight on the same data
ew = run_optimization(returns=mu, covariance=Sigma, objective="equal_weight")
print(f"Vol comparison — MinVar: {result.volatility:.4f}  |  EqualWgt: {ew.volatility:.4f}")
assert result.volatility <= ew.volatility + 1e-6, "min_variance should not exceed equal-weight vol"
print("Assertion passed: min_variance ≤ equal_weight volatility.")